In [1]:
!pip install -q streamlit pyngrok langchain langchain-community langchain-huggingface chromadb sentence-transformers pypdf transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 57.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 75.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 6.9 

In [2]:
from pyngrok import ngrok


NGROK_AUTH_TOKEN = "3KWz3pbM7VY7599kIJF5BRWajKd_6tdy9YjVETf494UdRdGSV"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [3]:
import os

os.makedirs("/kaggle/working/data", exist_ok=True)
os.makedirs("/kaggle/working/vector_store", exist_ok=True)

In [4]:
!pip install -q langchain-text-splitters

In [5]:
import os
import shutil
import glob
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


/tmp/ipykernel_48/1590520759.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [6]:
# 1. البحث الأوتوماتيكي عن أي ملف PDF ونسخه لفولدر المشروع
WORKING_DATA_PATH = "/kaggle/working/data"
os.makedirs(WORKING_DATA_PATH, exist_ok=True)

# البحث في كل ملفات كاجل المرفوعة ونسخها
print("Searching for PDF files...")
for pdf_file in glob.glob("/kaggle/input/**/*.pdf", recursive=True):
    shutil.copy(pdf_file, WORKING_DATA_PATH)
    print(f"Copied: {pdf_file.split('/')[-1]} to working directory.")

# 2. Define configuration and paths
DATA_PATH = WORKING_DATA_PATH
CHROMA_PATH = "/kaggle/working/vector_store"

# Using a multilingual embedding model suitable for both Arabic and English texts
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-small"

def initialize_vector_database():
    """Reads PDFs, splits them into chunks, and stores embeddings in ChromaDB."""
    
    # Check if there are any PDF files in the directory
    pdf_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.pdf')]
    if not pdf_files:
        print(f"Warning: No PDF files found in '{DATA_PATH}'.")
        print("Please check the upload process again.")
        return None

    # 3. Load PDF documents
    print(f"\nFound {len(pdf_files)} PDF(s). Loading documents...")
    loader = PyPDFDirectoryLoader(DATA_PATH)
    documents = loader.load()
    print(f"Successfully loaded {len(documents)} pages in total.")

    # 4. Split text into manageable chunks for the LLM context window
    print("Splitting documents into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Generated {len(chunks)} text chunks.")

    # 5. Initialize HuggingFace embeddings on GPU
    print(f"Downloading/Loading embedding model: '{EMBEDDING_MODEL_NAME}'...")
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_NAME,
        model_kwargs={'device': 'cuda'}  # Ensure it runs on the T4 GPU
    )

    # 6. Create Chroma vector store and save it to disk
    print("Creating Chroma vector database...")
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=CHROMA_PATH
    )
    
    print(f"\nSuccess! Vector database created and persisted at: {CHROMA_PATH}")
    return vector_db

# Execute the pipeline
db = initialize_vector_database()

Searching for PDF files...
Copied: s41598-025-29518-w_reference.pdf to working directory.
Copied: 1.10_OS-UN-J-25-3-2019-f.pdf to working directory.
Copied: modeling-and-simulation-of-egyptsat-1-satellite-system-5bgo0x1dr7.pdf to working directory.
Copied: 8- satellite engineering model lab.pdf to working directory.
Copied: introduction to brilliant remote sensing labs company brs-labs.pdf to working directory.
Copied: Egyptian-Space-Agency-EgSA-Concurrent-Design-Facility-Starion-whitepaper-2024.pdf to working directory.
Copied: cansat_brochure_final.pdf to working directory.
Copied: Mahmoud.Ibrahim.pdf to working directory.
Copied: IMD-Summary-2022.pdf to working directory.
Copied: MMT_2017_Paper_33.pdf to working directory.
Copied: Exploring_the_relationship_between_space_weather_c.pdf to working directory.
Copied: LSEJ_Volume 38_Issue 2_Pages 1326-1422.pdf to working directory.
Copied: 20221206130720583.pdf to working directory.
Copied: s41598-025-29518-w_reference (1).pdf to workin

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Creating Chroma vector database...

Success! Vector database created and persisted at: /kaggle/working/vector_store


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. تحديد اسم الموديل 
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading LLM {MODEL_NAME} on GPU (this might take a few minutes)...")

# 2. إعدادات الضغط (Quantization) عشان يشتغل بكفاءة على T4 GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# 3. تحميل الموديل والـ Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

# 4. بناء الـ Pipeline بتاع الموديل
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,  
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
)
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

# 5. تصميم الـ Prompt 
prompt_template = """You are a highly professional AI assistant for the Egyptian Space Agency (EgSA). 
Answer the question based ONLY on the following context. If you cannot find the answer in the context, say "I don't have enough information in my database to answer this question." Do not make up answers.

Context:
{context}

Question: {input}

Answer:"""
prompt = PromptTemplate.from_template(prompt_template)

# 6. دمج الموديل مع قاعدة البيانات بالطريقة الحديثة (LCEL)
print("Connecting LLM to Vector Database...")
retriever = db.as_retriever(search_kwargs={"k": 3}) 

# دالة صغيرة لتنظيم النصوص اللي بترجع من قاعدة البيانات
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# بناء محرك الـ RAG
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("\n🚀 RAG System is completely ready! 🚀")

# 7. تجربة سريعة للسيستم
test_question = "What is the main topic of the document?"
print(f"\nTesting question: {test_question}")

# استدعاء الموديل
response = rag_chain.invoke(test_question)
print("\nModel Answer:")
print(response)

Loading LLM Qwen/Qwen2.5-1.5B-Instruct on GPU (this might take a few minutes)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p', 'temperature', 'repetition_penalty', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Connecting LLM to Vector Database...

🚀 RAG System is completely ready! 🚀

Testing question: What is the main topic of the document?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Model Answer:
You are a highly professional AI assistant for the Egyptian Space Agency (EgSA). 
Answer the question based ONLY on the following context. If you cannot find the answer in the context, say "I don't have enough information in my database to answer this question." Do not make up answers.

Context:
Objective (2.B) 
Objective (2.C) 
Objective (2.D) 
Objective (2.E) 
Objective (2.F)
Objective (3.A)
Objective (3.B)
Objective (3.C)
Objective (3.D)
Objective (4.A)
Objective (4.B)
Objective (4.C)
Objective (4.D)
Objective (4.E)
Objective (5.A)
Objective (5.B)
Objective (5.C)
National Climate Change Strategy (NC:C:S) 
Objectives
Sustainable Development Strategy (SDS 2030) Objectives 
(The white icons in the ﬁgure)
Figure  3 -1: Interlinkage between the goals related to climate change in Egypt›s Updated Sustainable 
Development Strategy and the main goals and objectives of the NCCS

Purpose of the MoU: 
1. Integration of efforts and scientific and practical cooperation. 
2. Exploit

## App.py

In [29]:
%%writefile app.py
import streamlit as st
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ==========================================
# 1. System Constants & Enterprise UI Config
# ==========================================
st.set_page_config(page_title="EgSA AI Copilot | Egyptian Space Agency", page_icon="🛰️", layout="wide")

st.markdown("""
    <style>
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    .stChatFloatingInputContainer {padding-bottom: 20px;}
    .stButton>button {border-radius: 6px; font-weight: bold; border: 1px solid #1E3A8A;}
    </style>
""", unsafe_allow_html=True)

# ==========================================
# 2. Secure Authentication Logic
# ==========================================
valid_users = {"admin": "admin123", "engineer": "EgSA2026"}

def check_password():
    def password_entered():
        if st.session_state["user_input"] in valid_users and st.session_state["pass_input"] == valid_users[st.session_state["user_input"]]:
            st.session_state["password_correct"] = True
            st.session_state["logged_in_user"] = st.session_state["user_input"]
            del st.session_state["pass_input"]
        else:
            st.session_state["password_correct"] = False

    if st.session_state.get("password_correct", False):
        return True

    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.image("https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Egyptian_Space_Agency_logo.png/220px-Egyptian_Space_Agency_logo.png", width=150)
        st.title("EgSA Secure Portal 🔐")
        st.markdown("Enter your credentials to access the internal AI Copilot.")
        st.text_input("Username", key="user_input")
        st.text_input("Password", type="password", key="pass_input")
        st.button("Authenticate", on_click=password_entered, use_container_width=True)
        
        if "password_correct" in st.session_state and not st.session_state["password_correct"]:
            st.error("🚫 Access Denied: Invalid username or password.")
            
    return False

if not check_password():
    st.stop()

# ==========================================
# 3. High-Performance RAG System (Cached)
# ==========================================
@st.cache_resource(show_spinner=False)
def load_rag_system():
    # 1. Vector Store & Retriever
    embeddings = HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small", 
        model_kwargs={'device': 'cuda'}
    )
    db = Chroma(persist_directory="/kaggle/working/egsa_vector_db", embedding_function=embeddings)
    retriever = db.as_retriever(search_kwargs={"k": 5}) # Balanced for speed and accuracy
    
    # 2. LLM Quantization Config
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, 
        bnb_4bit_compute_dtype=torch.float16, 
        bnb_4bit_quant_type="nf4", 
        bnb_4bit_use_double_quant=True
    )
    
    # 3. Model & Tokenizer
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct", 
        quantization_config=quant_config, 
        device_map="auto"
    )
    
    # Force the model to stop exactly when the answer is done (Prevents hashtags/hallucinations)
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|im_end|>")
    ]
    
    # 4. Pipeline Optimization for SPEED and STRICTNESS
    pipe = pipeline(
        "text-generation", 
        model=model, 
        tokenizer=tokenizer, 
        max_new_tokens=384, 
        do_sample=False,        # Greedy decoding: FASTEST & strictly prevents hallucinations
        repetition_penalty=1.1, # Prevents word loops
        return_full_text=False,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id
    )
    llm = HuggingFacePipeline(pipeline=pipe)
    
    # 5. Professional Bilingual Prompt
    prompt = PromptTemplate.from_template("""<|im_start|>system
You are a highly professional, strictly formal AI assistant for the Egyptian Space Agency (EgSA). 
You are an expert in space engineering, satellite operations, and space research.

CRITICAL RULES:
1. You MUST answer in the EXACT SAME LANGUAGE as the user's question (e.g., Arabic for Arabic, English for English).
2. Maintain a highly scientific, direct, and professional tone. NEVER use emojis, hashtags, or conversational fillers.
3. Base your answer ONLY on the provided Context. If the answer is not in the context, reply strictly with: "I don't have enough information based on the provided documents." (or its exact Arabic translation: "لا أملك معلومات كافية بناءً على الوثائق المتاحة.").

Context: 
{context}
<|im_end|>
<|im_start|>user
Question: {input}<|im_end|>
<|im_start|>assistant
""")
    
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
        
    rag_chain = ({"context": retriever | format_docs, "input": RunnablePassthrough()} | prompt | llm | StrOutputParser())
    return rag_chain

rag_chain = load_rag_system()

# ==========================================
# 4. Professional Sidebar & Dashboard
# ==========================================
with st.sidebar:
    st.image("https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Egyptian_Space_Agency_logo.png/220px-Egyptian_Space_Agency_logo.png", width=150)
    st.title("EgSA AI Copilot")
    st.success(f"User: {st.session_state['logged_in_user'].upper()} 👨‍🚀")
    st.markdown("---")
    st.info("Status: Secure Connection 🟢")
    st.markdown("**Core Developer:** Ahmed Ehab")
    st.markdown("---")
    
    col1, col2 = st.columns(2)
    with col1:
        if st.button("Clear Chat", use_container_width=True):
            st.session_state.messages = []
            st.rerun()
    with col2:
        if st.button("Logout", use_container_width=True):
            st.session_state["password_correct"] = False
            st.rerun()

# ==========================================
# 5. Main Chat Interface
# ==========================================
st.title("EgSA Technical AI Assistant 🚀")
st.markdown("Welcome to the secure environment. You can query satellite manuals, space laws, and research papers (e.g., EgyptSat-1 performance studies).")

if "messages" not in st.session_state:
    st.session_state.messages = []

# Render chat history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Handle new user input
if prompt := st.chat_input("Enter your query here... (e.g., What are the ML models used in the EgyptSat-1 study?)"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Processing documents & formulating response... 🛰️"):
            # Generate response
            response = rag_chain.invoke(prompt)
            # Cleanup any weird artifacts just in case
            response = response.replace("<|im_end|>", "").strip()
            st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})

Overwriting app.py


In [30]:
import subprocess
from pyngrok import ngrok
import time

# 1. إغلاق أي روابط أو سيرفرات قديمة شغالة
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"])
time.sleep(2)

# 2. فتح رابط جديد للواجهة على بورت 8501
public_url = ngrok.connect(8501)
print(f"🚀 واجهة المشروع الخاصة بك جاهزة! اضغط على هذا الرابط للدخول:")
print(public_url.public_url)

# 3. تشغيل Streamlit في الخلفية بالطريقة الصحيحة المدعومة في كاجل
print("⏳ جاري إطلاق الواجهة... (برجاء فتح الرابط بعد 5 ثوانٍ)")
process = subprocess.Popen(["streamlit", "run", "app.py"])

  Stopping...
🚀 واجهة المشروع الخاصة بك جاهزة! اضغط على هذا الرابط للدخول:
https://skimmed-chest-unwitting.ngrok-free.dev
⏳ جاري إطلاق الواجهة... (برجاء فتح الرابط بعد 5 ثوانٍ)




2026-10-11 20:06:08.729 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://136.107.95.57:8501



/kaggle/working/app.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2792.09it/s]
/kaggle/working/app.py:70: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(persist_directory="/kaggle/working/egsa_vector_db", embedding_function=embeddings)
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 298.94it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'repe

In [17]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# مسار جديد تماماً لقاعدة البيانات لتجنب مشكلة الـ Read-only
NEW_CHROMA_PATH = "/kaggle/working/egsa_vector_db"
DATA_PATH = "/kaggle/working/data"

print("⏳ جاري قراءة الملفات وبناء قاعدة البيانات الجديدة (قد يستغرق بعض الوقت)...")
loader = PyPDFDirectoryLoader(DATA_PATH)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-small", model_kwargs={'device': 'cuda'})

# حفظ قاعدة البيانات في المسار الجديد
db = Chroma.from_documents(chunks, embeddings, persist_directory=NEW_CHROMA_PATH)
print(f"🚀 Success! تم معالجة {len(docs)} صفحة وبناء العقل الجديد بنجاح.")

⏳ جاري قراءة الملفات وبناء قاعدة البيانات الجديدة (قد يستغرق بعض الوقت)...


/kaggle/working/app.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🚀 Success! تم معالجة 526 صفحة وبناء العقل الجديد بنجاح.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2775.54it/s]
/kaggle/working/app.py:45: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 294.10it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'temperature', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precede